# Gap-filling metrics: one hold-out run

Explores one run of `find_best_model_holdout.py` (`performance_holdout/<gauge>/<run label>/`):
which model wins each selection metric, how the metrics agree, and how the winners look
against the observed flow, inside and outside the hidden block.

What it does not do: the referees (flow signatures, the day-by-day exceedance F1) are not
built yet, so these plots are for inspection, not a verdict. With few realizations per
model, small differences between winners are noise.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# The repo root holds the modules; the notebook runs with it as the working directory
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import find_best_model_holdout as fbh
import holdout_results as hr

warnings.filterwarnings("ignore", category=RuntimeWarning)  # overflow in poor parameter sets

GRID, SURFACE, INK = "#e1e0d9", "#fcfcfb", "#3b3b38"
SERIES_5 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # categorical slots 1-5, fixed order


def style(fig, **layout):
    """The look of best_model_per_metric.ipynb: light surface, grid, horizontal legend on top."""
    fig.update_xaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
    fig.update_layout(template="simple_white", plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
                      font=dict(color=INK), margin=dict(t=110, l=70, r=30, b=60),
                      legend=dict(orientation="h", x=0, y=1.03, xanchor="left", yanchor="bottom",
                                  font=dict(size=11)), **layout)
    return fig


# ## Settings

def newest_finished_run(root):
    """The newest run folder under root (by run_info.json time) whose status is "finished"."""
    finished = []
    for path in root.glob("*/*/run_info.json"):
        try:
            if json.loads(path.read_text()).get("status") == "finished":
                finished.append(path)
        except (OSError, ValueError):
            continue
    if not finished:
        raise FileNotFoundError(f"No finished run under {root}")
    return max(finished, key=lambda p: p.stat().st_mtime).parent


# Run folder: the newest finished run by default. To pick one, set it directly, e.g.
# RUN_DIR = fbh.PERFORMANCE_DIR / "07351750" / "smoke_07351750"
RUN_DIR = newest_finished_run(fbh.PERFORMANCE_DIR)

# Metrics to follow: five new ones, then NSE and KGE as the reference
CHOSEN_METRICS = ["KGE_NP", "KGE_TA_p5_p95", "KGE_TA_p5_p95_log", "PNSE_p5_p95", "JDKGE",
                  "NSE", "KGE"]
MODEL_FOR_SCATTER = None      # None = the model that wins the first chosen metric with a winner
X_METRIC, Y_METRIC = None, None  # None = the first two chosen metrics
LOG_Y = True                  # log y-axis on the hydrographs
SKIP_SNOW = False             # not recorded in run_info; this is the default the runs used;
                              # change it only if the run was started with a different value

In [ ]:
# ## Load the run

run = hr.load_run(RUN_DIR)
run_info, summary, tables = run["run_info"], run["summary"], run["tables"]
METRICS = list(fbh.SUMMARY_METRICS)  # the 15 metrics with a best value in summary.parquet

unknown = [m for m in CHOSEN_METRICS if m not in METRICS]
if unknown:
    raise ValueError(f"CHOSEN_METRICS has unknown names {unknown}; choose from {METRICS}")

print(f"Run folder: {RUN_DIR}")
for key in ["gauge", "start_year", "end_year", "warmup_years", "holdout_mode", "holdout_years",
            "holdout_position", "holdout_intervals", "extreme_date", "extreme_value",
            "models_evaluated", "models_removed", "models_skipped_no_temperature",
            "n_realizations", "seed", "elapsed_seconds", "git_commit", "git_dirty",
            "nonpositive_obs", "status"]:
    if key in run_info:
        print(f"  {key}: {run_info[key]}")
if run_info.get("status") != "finished":
    print(f"WARNING: run status is {run_info.get('status')!r}, not 'finished'; results may be incomplete")

In [ ]:
# ## Overview
#
# Share of the successful realizations (all models pooled) where each metric is NaN,
# realizations per model, and how often the new metrics hit an unexpected exception
# (`metrics_error`).

shares = pd.Series({m: hr.nan_share(tables, m) for m in METRICS})
fig = go.Figure(go.Bar(x=shares.index, y=shares.values, marker_color=SERIES_5[0],
                       hovertemplate="%{x}: %{y:.1%} NaN<extra></extra>"))
style(fig, title="Share of successful realizations with a NaN value", height=400,
      yaxis=dict(title="NaN share", range=[0, 1], tickformat=".0%")).show()

display(summary[["name", "n_ok", "n_failed"]])
n_metrics_error = sum(int(t["metrics_error"].notna().sum())
                      for t in tables.values() if "metrics_error" in t.columns)
print(f"metrics_error set in {n_metrics_error} realization(s)")

In [ ]:
# ## Best model per metric

best = hr.best_model_per_metric(summary, METRICS)
rows = []
for metric, b in best.items():
    rows.append({"metric": metric,
                 "model": "none (all NaN)" if b["model"] is None else b["model"],
                 "value": b["value"],
                 "second model": "none" if b["second_model"] is None else b["second_model"],
                 "second value": b["second_value"]})
display(pd.DataFrame(rows).set_index("metric"))

winners = {metric: b["model"] for metric, b in best.items()}
for model, won in hr.shared_winner_groups(winners).items():
    print(f"model {model} wins: {', '.join(won)}")
no_winner = [metric for metric, model in winners.items() if model is None]
if no_winner:
    print(f"no winner (all NaN): {', '.join(no_winner)}")

In [ ]:
# ## Rank correlation between the metrics
#
# Spearman correlation over all successful realizations of all models pooled. Blank cells:
# a metric that is NaN everywhere.

corr = hr.metric_rank_correlation(tables, METRICS)
labels = corr.map(lambda v: "" if pd.isna(v) else f"{v:.2f}")
fig = go.Figure(go.Heatmap(z=corr.to_numpy(), x=corr.columns, y=corr.index, zmin=-1, zmax=1,
                           colorscale="RdBu", text=labels.to_numpy(), texttemplate="%{text}",
                           hoverongaps=False))
style(fig, title="Spearman rank correlation between metrics", height=700,
      yaxis=dict(autorange="reversed")).show()

In [ ]:
# ## Metric against metric
#
# All successful realizations of one model, with the winning realization of each chosen
# metric marked (winners from other models are marked with their own values).

x_metric = X_METRIC or CHOSEN_METRICS[0]
y_metric = Y_METRIC or CHOSEN_METRICS[1]
scatter_model = MODEL_FOR_SCATTER
if scatter_model is None:
    scatter_model = next((best[m]["model"] for m in CHOSEN_METRICS if best[m]["model"] is not None), None)

if scatter_model is None:
    print("No chosen metric has a winner: nothing to plot")
else:
    table = tables[scatter_model]
    ok = table[table["error"].isna()]
    fig = go.Figure(go.Scatter(x=ok[x_metric], y=ok[y_metric], mode="markers",
                               name=f"model {scatter_model} realizations",
                               marker=dict(color="#b5b4ad", size=8), text=ok.index,
                               hovertemplate="realization %{text}<br>%{x:.3f}, %{y:.3f}<extra></extra>"))
    for i, metric in enumerate(CHOSEN_METRICS):
        model = best[metric]["model"]
        if model is None:
            print(f"{metric}: no winner (all NaN), not marked")
            continue
        row = hr.best_row(tables[model], metric)
        fig.add_trace(go.Scatter(x=[row[x_metric]], y=[row[y_metric]], mode="markers",
                                 name=f"best {metric}: model {model} r{row.name}",
                                 marker=dict(size=13, symbol="star",
                                             color=SERIES_5[i % len(SERIES_5)])))
    style(fig, title=f"{y_metric} against {x_metric}, model {scatter_model}",
          xaxis_title=x_metric, yaxis_title=y_metric, height=520).show()

In [ ]:
# ## Re-simulation of the winners
#
# The winning realization of each chosen metric is simulated again from its saved
# parameters, with the data rebuilt by `prepare_data` from the arguments in
# `run_info.json`. The check below recomputes NSE on the masked frame and compares it with
# the stored value.

# No forcing hash exists yet: a changed forcing file is only caught by the NSE check below.
# prepare_data may download again if the stored daily file does not cover the full period
# (this happened for gauge 07030500, whose record starts in 2001).


def rebuild_data(run_info, skip_snow):
    """The data dict of the run: prepare_data with the arguments main() passed (from run_info)."""
    return fbh.prepare_data(run_info["gauge"], run_info["start_year"], run_info["end_year"],
                            run_info["warmup_years"], run_info["et_method"], skip_snow=skip_snow,
                            precip_source=run_info["precip_source"],
                            nasa_sampling=run_info["nasa_sampling"],
                            holdout_mode=run_info["holdout_mode"],
                            holdout_years=run_info["holdout_years"],
                            holdout_position=run_info["holdout_position"])


def row_params(model, row):
    """The parameters of a table row, as _run_one receives them (name -> float)."""
    return {name: float(row[name]) for name in fbh.marrmot.MODEL_REGISTRY[model]["param_names"]}


def resimulate(model, row, data):
    """(full, masked): the Q / Qobs frame as _run_one builds it, before and after hiding the block."""
    q = fbh.simulate_series(model, row_params(model, row), data)  # floored at Q_FLOOR
    full = pd.DataFrame({"Q": q, "Qobs": data["Qobs"]}, index=data["dates"])
    masked = fbh.hb.mask_hidden(full, data["hidden"]) if data["hidden"].any() else full
    return full, masked


def resimulate_winners(metrics, best, tables, data):
    """{metric: {"model", "row", "frame" (masked), "full"}} for each metric with a winner."""
    out = {}
    for metric in metrics:
        model = best[metric]["model"]
        if model is None:
            print(f"{metric}: no winner (all NaN), skipped")
            continue
        row = hr.best_row(tables[model], metric)
        if row is None:
            print(f"{metric}: model {model} has no valid row, skipped")
            continue
        full, masked = resimulate(model, row, data)
        out[metric] = {"model": model, "row": row, "frame": masked, "full": full}
    return out


def check_nse(resims, run_info):
    """OK / WARNING per winner: NSE of the re-simulated masked frame against the stored NSE."""
    mode = {"nonpositive": run_info["nonpositive_obs"], "offset_frac": run_info["obs_offset_frac"]}
    for metric, w in resims.items():
        stored = float(w["row"]["NSE"])
        try:
            again = float(fbh.pm.nse(w["frame"], "Q", "Qobs", **mode))
        except ValueError as exc:
            again = float("nan")
            print(f"  ({metric}: NSE could not be recomputed: {exc})")
        label = f"{metric} winner (model {w['model']}, realization {w['row'].name})"
        if np.isnan(stored) and np.isnan(again):
            print(f"OK {label}: NSE is NaN in both")
        elif abs(again - stored) <= 1e-9 * abs(stored):
            print(f"OK {label}: NSE {again:.12g} equals the stored value")
        else:
            print(f"WARNING {label}: NSE {again!r} vs stored {stored!r}; the re-simulation or "
                  "the forcing differs from the run")


data = rebuild_data(run_info, SKIP_SNOW)
resims = resimulate_winners(CHOSEN_METRICS, best, tables, data)
check_nse(resims, run_info)

In [ ]:
# ## Hydrograph
#
# Observed flow (training days in black, hidden days in orange) and one line per winning
# realization (a realization that wins several metrics is drawn once). The hold-out block is
# shaded. Click a legend entry to hide or show it; use the range slider to move along.

obs = pd.Series(data["Qobs"], index=data["dates"])
hidden = pd.Series(data["hidden"], index=data["dates"])
intervals = run_info.get("holdout_intervals") or []

# One line per distinct (model, realization), labeled with the metrics it wins
lines = {}
for metric, w in resims.items():
    key = (w["model"], w["row"].name)
    lines.setdefault(key, {"sim": w["full"]["Q"], "metrics": []})["metrics"].append(metric)


def hydrograph(x_range=None, title=""):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=obs.index, y=obs.where(~hidden), name="Observed (training days)",
                             line=dict(color=INK, width=1.2)))
    fig.add_trace(go.Scatter(x=obs.index, y=obs.where(hidden), name="Observed (hidden days)",
                             line=dict(color="#c0392b", width=1.2)))
    for i, ((model, realization), line) in enumerate(lines.items()):
        fig.add_trace(go.Scatter(x=line["sim"].index, y=line["sim"],
                                 line=dict(width=1, color=SERIES_5[i % len(SERIES_5)]),
                                 name=f"model {model} r{realization} ({', '.join(line['metrics'])})"))
    for start, end in intervals:
        fig.add_vrect(x0=start, x1=end, fillcolor="#8a8984", opacity=0.12, line_width=0,
                      layer="below")
    style(fig, title=title, height=600, hovermode="x unified")
    fig.update_yaxes(title_text="Flow (mm/day)", type="log" if LOG_Y else "linear")
    fig.update_xaxes(rangeslider_visible=True, range=x_range)
    return fig


hydrograph(title=f"Gauge {run_info['gauge']}: observed and winners").show()
if intervals:
    zoom = [intervals[0][0], intervals[-1][1]]
    hydrograph(zoom, f"Gauge {run_info['gauge']}: the hold-out block").show()
else:
    print("No hold-out block in this run: no zoomed figure")

In [ ]:
# ## Flow-duration curves
#
# Flow against exceedance probability, for the training days and for the hidden days
# (observed days only, the winners on the same days). These are percentiles of the sorted
# flows, so they carry no timing. Dashed lines: the 5/95 and 2/98 percent anchors; shaded:
# the default low and high blocks (the `pdi_low` and `pdi_high` percentile lists).

def exceedance_curve(values):
    """Exceedance probability (%) and flow, sorted from high to low (Weibull positions)."""
    q = np.sort(values)[::-1]
    return 100 * np.arange(1, len(q) + 1) / (len(q) + 1), q


def flow_duration(days, title):
    fig = go.Figure()
    series = [("Observed", obs[days], INK)] + \
             [(f"model {m} r{r} ({', '.join(line['metrics'])})", line["sim"][days],
               SERIES_5[i % len(SERIES_5)])
              for i, ((m, r), line) in enumerate(lines.items())]
    for name, values, color in series:
        values = values.to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        n_zero = int((values <= 0).sum())
        if n_zero:
            print(f"{title} - {name}: {n_zero} zero (or negative) value(s) not drawn on the log axis")
        x, y = exceedance_curve(values[values > 0])
        fig.add_trace(go.Scatter(x=x, y=y, name=name, line=dict(width=1.4, color=color)))
    # Anchors: percentile p sits at exceedance 100 - p (5/95 and 2/98 give the same lines)
    for p in (2, 5, 95, 98):
        fig.add_vline(x=p, line=dict(color="#8a8984", width=1, dash="dash"))
    low, high = fbh.gm.DEFAULT_LOW_BLOCK, fbh.gm.DEFAULT_HIGH_BLOCK
    fig.add_vrect(x0=100 - max(low), x1=100 - min(low), fillcolor="#2a78d6", opacity=0.08, line_width=0)
    fig.add_vrect(x0=100 - max(high), x1=100 - min(high), fillcolor="#eb6834", opacity=0.08, line_width=0)
    style(fig, title=title, height=520)
    fig.update_xaxes(title_text="Exceedance probability (%)", range=[0, 100])
    fig.update_yaxes(title_text="Flow (mm/day)", type="log")
    return fig


observed = obs.notna()
flow_duration(observed & ~hidden, "Flow-duration curve: training days").show()
if hidden.any():
    flow_duration(observed & hidden, "Flow-duration curve: hidden days").show()
else:
    print("No hidden days in this run: no hidden-day curve")

In [ ]:
# ## Metric values of the winners

values = pd.DataFrame(
    [w["row"][METRICS].astype(float).rename(f"{metric}: model {w['model']} r{w['row'].name}")
     for metric, w in resims.items()])
display(values)

## Referees (not built yet)

Flow signatures and the day-by-day exceedance F1 on the hidden days will be added here.
Until then, no conclusion about which metric is better should be drawn from the plots
above.